### 🔢 Study Session 4 — Compute Adjusted R²

While R² tells us how much of the variation in the target is explained by the model,  
it **always increases** when you add more predictors — even if they add no real value.

**Adjusted R²** corrects for that by penalizing extra predictors.

$$
\text{Adjusted R}^2 = 1 - (1 - R^2)\frac{n - 1}{n - p - 1}
$$

- \( n \) = number of rows (observations)  
- \( p \) = number of predictors (features in X)  

Use it to judge whether adding new variables genuinely improves your model fit.


In [1]:
# --- 1. Import libraries ---
import os
import sys
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

# Manually define the correct project root path based on your previous output
CORRECT_PROJECT_ROOT = "/home/rtackett/projects/Masters-level-DIY-Data-Science-Curriculum-ai-Era-/ds-zero-to-one"

# Set CWD
try:
    os.chdir(CORRECT_PROJECT_ROOT)
    print(f"✅ CWD successfully set to: {os.getcwd()}")

    # Add to sys.path for module imports (src.helpers)
    if CORRECT_PROJECT_ROOT not in sys.path:
        sys.path.append(CORRECT_PROJECT_ROOT)
        print("✅ Added project root to sys.path.")

except FileNotFoundError:
    print("❌ CRITICAL ERROR: The manually defined project path does not exist.")
    sys.exit(1)

# --- 2. Load dataset saved previously above from seaborn's GitHub mirror ---
# NOTE: Path adjusted to be relative to the Project Root CWD
df = duckdb.query("""
    SELECT * FROM read_csv_auto('data/processed/tips_cleaned.csv')
""").df()

df.info

✅ CWD successfully set to: /home/rtackett/projects/Masters-level-DIY-Data-Science-Curriculum-ai-Era-/ds-zero-to-one
✅ Added project root to sys.path.


<bound method DataFrame.info of      bill_total_usd  tip_usd  gender  is_smoker   day    time  party_size  \
0             16.99     1.01  Female      False   Sun  Dinner           2   
1             10.34     1.66    Male      False   Sun  Dinner           3   
2             21.01     3.50    Male      False   Sun  Dinner           3   
3             23.68     3.31    Male      False   Sun  Dinner           2   
4             24.59     3.61  Female      False   Sun  Dinner           4   
..              ...      ...     ...        ...   ...     ...         ...   
239           29.03     5.92    Male      False   Sat  Dinner           3   
240           27.18     2.00  Female       True   Sat  Dinner           2   
241           22.67     2.00    Male       True   Sat  Dinner           2   
242           17.82     1.75    Male      False   Sat  Dinner           2   
243           18.78     3.00  Female      False  Thur  Dinner           2   

      tip_pct  
0    0.059447  
1    0.1605

In [4]:
import pandas as pd

# 1) Work on a copy
df2 = df.copy()

# 2) Normalize categorical text (handles typos/case like 'M', 'male ', 'YES', True/False, etc.)
def norm_gender(x):
    x = str(x).strip().lower()
    if x in {"m", "male"}: return "Male"
    if x in {"f", "female"}: return "Female"
    return pd.NA

def norm_smoker(x):
    s = str(x).strip().lower()
    if s in {"yes", "y", "true", "1"}: return "Yes"
    if s in {"no", "n", "false", "0"}: return "No"
    return pd.NA

df2["gender"] = df2["gender"].apply(norm_gender)
df2["is_smoker"] = df2["is_smoker"].apply(norm_smoker)

# 3) Lock categories so dummies are predictable even if a level is absent in this sample
df2["gender"] = pd.Categorical(df2["gender"], categories=["Female", "Male"])
df2["is_smoker"] = pd.Categorical(df2["is_smoker"], categories=["No", "Yes"])

# 4) Make dummies (drop_first=True => reference levels Female, No)
df_enc = pd.get_dummies(df2, columns=["gender", "is_smoker"], drop_first=True)

# 5) Guarantee expected columns exist (in case this subset lacks one level)
for col in ["gender_Male", "is_smoker_Yes"]:
    if col not in df_enc.columns:
        df_enc[col] = 0

# 6) Sanity checks
print("Dummy columns present:", [c for c in df_enc.columns if c.startswith(("gender_", "is_smoker_"))])
print("gender unique:", df2["gender"].unique())
print("is_smoker unique:", df2["is_smoker"].unique())


Dummy columns present: ['gender_Male', 'is_smoker_Yes']
gender unique: ['Female', 'Male']
Categories (2, object): ['Female', 'Male']
is_smoker unique: ['No', 'Yes']
Categories (2, object): ['No', 'Yes']


In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

X = df_enc[["bill_total_usd", "party_size", "gender_Male", "is_smoker_Yes"]]
y = df_enc["tip_usd"]

model = LinearRegression().fit(X, y)
y_pred = model.predict(X)

print("Intercept:", round(model.intercept_, 3))
for name, coef in zip(X.columns, model.coef_):
    print(f"{name}: {coef:.3f}")
print("R²:", round(r2_score(y, y_pred), 3))


Intercept: 0.722
bill_total_usd: 0.094
party_size: 0.180
gender_Male: -0.027
is_smoker_Yes: -0.084
R²: 0.469


## 🧠 Model Fit Summary

| Metric | Value | Plain-English Meaning |
|:--------|:-------|:----------------------|
| **R²** | 0.469 | About 47% of the variation in tips can be explained by the model. |
| **Adjusted R²** | 0.460 | After adjusting for the number of predictors, the model still explains about 46% of the variation — meaning extra variables didn’t add much more power. |

💡 **ELI5:**  
R² tells us *how much of the tipping pattern our model captures*.  
Adjusted R² says *“don’t brag too soon”* — it checks if those extra predictors really helped.

🧾 **In Plain Words:**  
Our model explains about **half of why tips go up or down**, but the rest depends on unpredictable human choices.


In [6]:
import numpy as np
from sklearn.metrics import r2_score

# R² from your model
r2 = r2_score(y, y_pred)

# n = number of rows, p = number of predictors
n = len(y)
p = X.shape[1]

adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print(f"R²: {r2:.3f}")
print(f"Adjusted R²: {adj_r2:.3f}")


R²: 0.469
Adjusted R²: 0.460
